In [6]:
import pandas as pd
import numpy as np
import spiceypy as spy
import plotly.graph_objects as go
from Utils import CanonicalUnits, GravitationalParameters

In [7]:
path_data = "../datos/sbdb_query_results_NEOS.csv"

neos = pd.read_csv(path_data)
neos["q"]=neos["a"]*(1-neos["e"])
neos = neos[neos["q"] < 1.30]
neos

,pdes,epoch_mjd,a,e,i,om,w,ma,q
0,433,60200,1.458,0.2228,10.83,304.29,178.91,222.76,1.133158
1,719,60200,2.636,0.5470,11.58,183.85,156.23,56.29,1.194108
2,887,60200,2.472,0.5709,9.40,110.42,350.47,238.76,1.060735
3,1036,60200,2.666,0.5329,26.69,215.50,132.48,276.42,1.245289
4,1221,60200,1.919,0.4357,11.88,171.32,26.65,123.53,1.082892
...,...,...,...,...,...,...,...,...,...
34323,2024 CP5,60200,2.588,0.6057,3.48,309.59,206.24,321.35,1.020448
34324,2024 CQ5,60350,2.180,0.8506,2.16,228.85,146.50,16.09,0.325692
34325,2024 CR5,60352,1.136,0.3680,17.82,141.95,100.20,302.45,0.717952
34326,2024 CS5,60353,1.121,0.4184,42.92,140.26,168.63,222.21,0.651974


In [8]:
orbital_elements = np.column_stack((np.array(neos['a']), np.array(neos['e']), np.array(neos['i']), np.array(neos['om']), np.array(neos['w']), np.array(neos['ma'])))
orbital_elements

array([[1.4580e+00, 2.2280e-01, 1.0830e+01, 3.0429e+02, 1.7891e+02,
        2.2276e+02],
       [2.6360e+00, 5.4700e-01, 1.1580e+01, 1.8385e+02, 1.5623e+02,
        5.6290e+01],
       [2.4720e+00, 5.7090e-01, 9.4000e+00, 1.1042e+02, 3.5047e+02,
        2.3876e+02],
       ...,
       [1.1360e+00, 3.6800e-01, 1.7820e+01, 1.4195e+02, 1.0020e+02,
        3.0245e+02],
       [1.1210e+00, 4.1840e-01, 4.2920e+01, 1.4026e+02, 1.6863e+02,
        2.2221e+02],
       [2.8210e+00, 6.6180e-01, 4.6800e+00, 1.8281e+02, 2.3497e+02,
        1.3025e+02]])

In [9]:
deg = np.pi/180
AU_m = 1.496e11 #m
M_sun = 1.9891e30
G = 6.67430e-11 # m^3 / (kg s^2)
year = 365.25*24*3600 #s
mu = CanonicalUnits().mu
grav_params = GravitationalParameters(mu=mu)

In [10]:
state_vectors = np.zeros((len(orbital_elements), 6))
for index, elements in enumerate(orbital_elements):
    a = elements[0]
    e = elements[1]
    i = elements[2]*deg
    Omega = elements[3]*deg
    w = elements[4]*deg
    M = elements[5]*deg
    q = a*(1-e)

    state_vector = spy.conics([q, e, i, Omega, w, M]+[0, mu], 0)
    state_vectors[index] = np.array(state_vector[:6])

In [11]:
state_vectors

array([[ 1.50564476e+00, -8.24086449e-01,  1.49155679e-01,
         1.48131740e+00,  4.01462703e+00,  6.66810679e-01],
       [-4.97321553e-01,  2.47466938e+00, -5.12774515e-01,
        -3.67123769e+00,  1.44378549e+00, -3.45684097e-01],
       [ 1.86254335e+00, -3.05239693e+00, -1.12659812e-01,
         1.39433630e+00,  2.01164521e+00, -3.32517926e-01],
       ...,
       [-8.27481539e-01,  6.47049584e-01,  1.55307411e-04,
        -1.66731661e+00, -5.86126182e+00,  1.81401780e+00],
       [-1.26385772e+00,  8.43129403e-01,  1.48471976e-01,
        -6.40888573e-01, -3.07389320e+00,  2.57901055e+00],
       [-3.19994846e+00, -3.05371847e+00,  2.36845340e-01,
         6.56921545e-01, -1.83756004e+00,  1.52884727e-01]])

In [15]:
xs = state_vectors[:,0]
ys = state_vectors[:,1]
zs = state_vectors[:,2]
# Add 1 to shift the mean of the Gaussian distribution

fig = go.Figure()
fig.add_trace(go.Histogram(x=xs, name='x'))
fig.add_trace(go.Histogram(x=ys, name='y'))
fig.add_trace(go.Histogram(x=zs, name='z'))
# Reduce opacity to see both histograms
fig.update_traces(opacity=0.75)
fig.update_xaxes(range=[-5, 5])
fig.show()

In [17]:
vxs = state_vectors[:,3]
vys = state_vectors[:,4]
vzs = state_vectors[:,5]
# Add 1 to shift the mean of the Gaussian distribution

fig = go.Figure()
fig.add_trace(go.Histogram(x=vxs, name='vx'))
fig.add_trace(go.Histogram(x=vys, name='vy'))
fig.add_trace(go.Histogram(x=vzs, name='vz'))
# Reduce opacity to see both histograms
fig.update_traces(opacity=0.75)
fig.update_xaxes(range=[-10, 10])
fig.show()